In [ ]:
import torch
import torch.nn as nn

Creating the first convolution block (to be used in double convolution)

In [ ]:
class Conv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        #Applying convolution over the input signal
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=3,padding=1)
        #normalizing activations of a convolution layer per conv2d output channel
        self.bn = nn.BatchNorm2d(out_ch)
        #Applying relu activation (relu(x) = max(0,x) only keeping +ve vals)
        self.relu = nn.ReLU()

    def forward(self,x):

        x=self.conv(x)
        x=self.bn(x)
        x = self.relu(x)

        return x

In [82]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.conv1 = Conv(in_ch,out_ch)
        self.conv2 = Conv(out_ch, out_ch)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)

        return x

In [ ]:
class DownSample(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        #Creating a maxpool layer to half the dimensions
        self.pool = nn.MaxPool2d(2)
        self.double_conv = DoubleConv(in_ch,out_ch)

    def forward(self,x):
        x = self.pool(x)
        x = self.double_conv(x)


        return x

In [ ]:
class Upsample(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        #creating transposed convolution layer to double dimensions (opposite to maxpooling)
        self.up = nn.ConvTranspose2d(in_ch,out_ch,kernel_size=2,stride=2)

        self.double_conv = DoubleConv(out_ch*2,out_ch)

    def forward(self,x,skip):
        x = self.up(x)
        #Here we are concatenating the skip dimensions with x along the channel dimensions
        x =torch.cat([skip,x], dim=1)
        x = self.double_conv(x)


        return x


In [ ]:
class UNet(nn.Module):
    def __init__(self,in_ch=3, out_ch=1):
        super().__init__()
        #In channels = 3 (RGB MODE) 


        self.inc = DoubleConv(in_ch,64)
        #From 3 rgb channels to output 64 feature channels
        #During encoding stage, channel depth keeps doubling to learn complex features and shapes
        #But spatial dimensions decrease
        self.down1 = DownSample(64,128)
        self.down2 = DownSample(128,256)
        self.down3 = DownSample(256,512)
        self.down4 = DownSample(512,1024)

# Upsampling doubles the spatial dimensions using transposde conv
# Increasing dimensions while halving the channel depth       
        self.up1 = Upsample(1024,512)
        self.up2 = Upsample(512,256)
        self.up3 = Upsample(256,128)
        self.up4 = Upsample(128,64)

        self.out = nn.Conv2d(64,out_ch,kernel_size=1)

#To understand what is happening at each stage, shape is printed directly after each step
    def forward(self,x):
        print(f"Input: {x.shape}")
        x1 = self.inc(x)
        print(f"Encoder 1: {x1.shape}")
        x2 = self.down1(x1)
        print(f"Encoder 2: {x2.shape}")
        x3 = self.down2(x2)
        print(f"Encoder 3: {x3.shape}")
        x4 = self.down3(x3)
        print(f"Encoder 4: {x4.shape}")
        x5 = self.down4(x4)
        print(f"Botteleneck: {x5.shape}")


        x = self.up1(x5,x4)
        print(f"Decoder 1: {x.shape}")
        x = self.up2(x, x3)
        print(f"Decoder 2: {x.shape}")
        x = self.up3(x, x2)
        print(f"Decoder 3: {x.shape}")
        x = self.up4(x, x1)
        print(f"Decoder 4: {x.shape}")

        x = self.out(x)
        print(f"Output: {x.shape}")
        return x

In [98]:
model = UNet()

x = torch.randn(1,3,256,256)

y = model(x)


Input: torch.Size([1, 3, 256, 256])
Encoder 1: torch.Size([1, 64, 256, 256])
Encoder 2: torch.Size([1, 128, 128, 128])
Encoder 3: torch.Size([1, 256, 64, 64])
Encoder 4: torch.Size([1, 512, 32, 32])
Botteleneck: torch.Size([1, 1024, 16, 16])
Decoder 1: torch.Size([1, 512, 32, 32])
Decoder 2: torch.Size([1, 256, 64, 64])
Decoder 3: torch.Size([1, 128, 128, 128])
Decoder 4: torch.Size([1, 64, 256, 256])
Output: torch.Size([1, 1, 256, 256])


In [99]:
model.eval()

UNet(
  (inc): DoubleConv(
    (conv1): Conv(
      (conv): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU()
    )
    (conv2): Conv(
      (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU()
    )
  )
  (down1): DownSample(
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (double_conv): DoubleConv(
      (conv1): Conv(
        (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU()
      )
      (conv2): Conv(
        (conv): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, af